In [1]:
import os
import warnings
from pathlib import Path
from typing import Any

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from model_config import LocalModel, RemoteModel  # noqa: E402
from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str_remote: str = RemoteModel.GPT_OSS_120B
model_str_local: str = LocalModel.MISTRAL_7B_INSTRUCT_V0_3_Q4_0

# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str_remote,
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),
    base_url=settings.OLLAMA_URL,
    temperature=0.0,
    model=model_str_local,
)

In [5]:
from uuid import uuid4

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

### Load Data

In [6]:
fp: str = "../../data/chelsea_transfer_news.pdf"

loader = PyPDFLoader(fp)
docs = loader.load()

# Split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=100
)
splits = text_splitter.split_documents(docs)

console.print(f"Number of chunks: {len(splits)}", style="info")

Number of chunks: 13

In [7]:
console.print(docs[0])

Document(
    metadata={
        'producer': 'Skia/PDF m138',
        'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
        'creationdate': '2025-07-25T18:28:43+00:00',
        'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans and 
contracts | Football News | Sky Sports',
        'moddate': '2025-07-25T18:28:43+00:00',
        'source': '../../data/chelsea_transfer_news.pdf',
        'total_pages': 12,
        'page': 0,
        'page_label': '1'
    },
    page_content='Friday 25 July 2025 14:39, UK\nChelsea transfer news, rumours and\ngossip: Live updates and 
latest on\ndeals, signings, loans and contracts\nLatest Chelsea news\xa0\nSort by: Latest Oldest\nIn full: Chelsea 
2025/26 Premier League fixtures\xa0\nTransfer Centre LIVE! Deals, rumours, news on your phone\nDownload the Sky 
Sports app for Chelsea transfers, analysis and FREE highlights from\nEVERY Premier League game\xa0\xa0\xa0View 
post\n24 Jul\n16:22 Simons to Chelsea? The key questions answered...\nChelsea have held talks over signing RB 
Leipzig forward Xavi Simons - but how\nmuch will they have to pay and who else want him?\xa0\nSky Germany’s Leipzig
reporter Philipp Hinze answers the key questions\naround the deal.\xa0\nKeep scrolling!\xa0\nUPDATE\nF o o t b al l
\n News Watch Scores & FixturesTables Transfers More\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: 
Live updates and latest on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 1/17'
)

In [8]:
console.print(splits[0])

Document(
    metadata={
        'producer': 'Skia/PDF m138',
        'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
        'creationdate': '2025-07-25T18:28:43+00:00',
        'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans and 
contracts | Football News | Sky Sports',
        'moddate': '2025-07-25T18:28:43+00:00',
        'source': '../../data/chelsea_transfer_news.pdf',
        'total_pages': 12,
        'page': 0,
        'page_label': '1'
    },
    page_content='Friday 25 July 2025 14:39, UK\nChelsea transfer news, rumours and\ngossip: Live updates and 
latest on\ndeals, signings, loans and contracts\nLatest Chelsea news\xa0\nSort by: Latest Oldest\nIn full: Chelsea 
2025/26 Premier League fixtures\xa0\nTransfer Centre LIVE! Deals, rumours, news on your phone\nDownload the Sky 
Sports app for Chelsea transfers, analysis and FREE highlights from\nEVERY Premier League game\xa0\xa0\xa0View 
post\n24 Jul\n16:22 Simons to Chelsea? The key questions answered...\nChelsea have held talks over signing RB 
Leipzig forward Xavi Simons - but how\nmuch will they have to pay and who else want him?\xa0\nSky Germany’s Leipzig
reporter Philipp Hinze answers the key questions\naround the deal.\xa0\nKeep scrolling!\xa0\nUPDATE\nF o o t b al l
\n News Watch Scores & FixturesTables Transfers More\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: 
Live updates and latest on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 1/17'
)

In [9]:
splits[0].metadata["source"]  # source
splits[0].metadata["page_label"]  # page_label

'1'

### Indexing

In [10]:
collection_name: str = "cfc_transfer_news"

emb_model = OllamaEmbeddings(
    model=LocalModel.MXBAI_EMBED_LARGE,
)
emb = emb_model.embed_documents("Hello world")
emb_size: int = len(emb[0])
console.print(f"Embedding size: {emb_size}", style="info")

client = QdrantClient(url=settings.QDRANT_URL)
collection_exists_flag: bool = client.collection_exists(collection_name=collection_name)

if not collection_exists_flag:
    client = QdrantClient(url=settings.QDRANT_URL)
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
    )

    # Vector store
    vector_store: QdrantVectorStore = QdrantVectorStore.from_documents(
        documents=splits,
        embedding=emb_model,
        collection_name=collection_name,
        ids=[str(uuid4()) for _ in range(len(splits))],
    )
else:
    vector_store = QdrantVectorStore.from_existing_collection(
        embedding=emb_model,
        collection_name=collection_name,
        url=settings.QDRANT_URL,
    )

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Embedding size: 1024

In [11]:
collection_name: str = "cfc_transfer_news"

emb_model = OllamaEmbeddings(
    model=LocalModel.MXBAI_EMBED_LARGE.value,
)
emb = emb_model.embed_documents("Hello world")
emb_size: int = len(emb[0])
console.print(f"Embedding size: {emb_size}", style="info")

client = QdrantClient(url=settings.QDRANT_URL)
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
)

# Vector store
vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=emb_model,
    collection_name=collection_name,
    ids=[str(uuid4()) for _ in range(len(splits))],
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Embedding size: 1024

## Query Transformation

- Query transformations are a set of approaches focused on re-writing and / or modifying questions for retrieval.

<br>

### 1. Multi Query

- `Multi Query` uses an LLM to generate multiple versions of a single query.

- Each new query is used to perform a separate search.

- The unique documents from all searches are then combined to create a comprehensive set of results.

[![image.png](https://i.postimg.cc/50BsnRfp/image.png)](https://postimg.cc/DJzQz5Jb)

In [12]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Multi Query: Different Perspectives
template: str = """
<system>
    You are an AI language model assistant. 
    <role>
    Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector 
    database.
    </role>
    <instructions>
    By generating multiple perspectives on the user question, your goal is to help the user overcome some of the 
    limitations of the distance-based similarity search. Provide these alternative questions separated by newlines. 
    <original_question>
    {question}
    </original_question>
    </instructions>
</system>
"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    | remote_llm  # local_llm
    | StrOutputParser()
    | (lambda x: x.strip().split("\n"))
)

In [13]:
from IPython.display import Markdown, display

display(Markdown("### Hello"))

### Hello

In [14]:
question: str = "What players are likely to exit Chelsea?"
response = generate_queries.invoke({"question": question})

console.print(question, style="info")
console.print(response)

What players are likely to exit Chelsea?

[
    'Which Chelsea players are expected to leave the club?  ',
    'Who are the likely departures from Chelsea this season?  ',
    'Which members of the Chelsea squad might exit in the upcoming transfer window?  ',
    'What footballers are predicted to move on from Chelsea?  ',
    'Which Chelsea players are rumored to be on their way out?'
]

In [15]:
from langchain.load import dumps, loads


def get_unique_union(documents: list[list[Any]]) -> list[Any]:
    """
    Get the unique union of lists of documents.

    Parameters
    ----------
    documents : list[list[Any]]
        A list of lists containing documents.

    Returns
    -------
    list[Any]
        A list containing unique documents.
    """
    # Flatten list of lists and convert each Document to a string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))

    return [loads(doc) for doc in unique_docs]


# Retrieve
question: str = "What players are likely to exit Chelsea?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
retr_docs = retrieval_chain.invoke({"question": question})
len(retr_docs)

/var/folders/vv/g_5scsqs6fj18dr1q_bww19r0000gn/T/ipykernel_73267/1780963150.py:22: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


4

In [16]:
console.print(retr_docs)

[
    Document(
        metadata={
            'producer': 'Skia/PDF m138',
            'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
            'creationdate': '2025-07-25T18:28:43+00:00',
            'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans 
and contracts | Football News | Sky Sports',
            'moddate': '2025-07-25T18:28:43+00:00',
            'source': '../../data/chelsea_transfer_news.pdf',
            'total_pages': 12,
            'page': 4,
            'page_label': '5',
            '_id': 'dd72468c-201a-49e7-98d0-3ffc51d54e8d',
            '_collection_name': 'cfc_transfer_news'
        },
        page_content='actually solid.\n"The club’s poor season casts a heavy shadow over all the players. After a 
very\nstrong season the year before, Xavi missed the chance to take the next step\nin his 
development."\nAdvertisement\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: Live updates and latest 
on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 5/17'
    ),
    Document(
        metadata={
            'producer': 'Skia/PDF m138',
            'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
            'creationdate': '2025-07-25T18:28:43+00:00',
            'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans 
and contracts | Football News | Sky Sports',
            'moddate': '2025-07-25T18:28:43+00:00',
            'source': '../../data/chelsea_transfer_news.pdf',
            'total_pages': 12,
            'page': 0,
            'page_label': '1',
            '_id': 'd45133c3-cc66-4b19-b12e-1934ca3be59c',
            '_collection_name': 'cfc_transfer_news'
        },
        page_content='Friday 25 July 2025 14:39, UK\nChelsea transfer news, rumours and\ngossip: Live updates and 
latest on\ndeals, signings, loans and contracts\nLatest Chelsea news\xa0\nSort by: Latest Oldest\nIn full: Chelsea 
2025/26 Premier League fixtures\xa0\nTransfer Centre LIVE! Deals, rumours, news on your phone\nDownload the Sky 
Sports app for Chelsea transfers, analysis and FREE highlights from\nEVERY Premier League game\xa0\xa0\xa0View 
post\n24 Jul\n16:22 Simons to Chelsea? The key questions answered...\nChelsea have held talks over signing RB 
Leipzig forward Xavi Simons - but how\nmuch will they have to pay and who else want him?\xa0\nSky Germany’s Leipzig
reporter Philipp Hinze answers the key questions\naround the deal.\xa0\nKeep scrolling!\xa0\nUPDATE\nF o o t b al l
\n News Watch Scores & FixturesTables Transfers More\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: 
Live updates and latest on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 1/17'
    ),
    Document(
        metadata={
            'producer': 'Skia/PDF m138',
            'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
            'creationdate': '2025-07-25T18:28:43+00:00',
            'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans 
and contracts | Football News | Sky Sports',
            'moddate': '2025-07-25T18:28:43+00:00',
            'source': '../../data/chelsea_transfer_news.pdf',
            'total_pages': 12,
            'page': 8,
            'page_label': '9',
            '_id': '968599a1-33f4-4dbe-b3b5-9b48f1152337',
            '_collection_name': 'cfc_transfer_

In [17]:
from pydantic import BaseModel, Field


class ResponseSources(BaseModel):
    source: str = Field(..., description="The name of the document")
    page_label: str = Field(..., description="The page label of the document")


class LLMResponse(BaseModel):
    answer: str = Field(..., description="The answer generated by the LLM")
    sources: list[ResponseSources] = Field(
        default_factory=list,
        description="List of sources used to generate the answer",
    )


def format_docs(docs: list[Any]) -> str:
    """Format a list of documents into a string representation."""
    return "\n\n".join(
        f"(content: {doc.page_content} \n\n source: {doc.metadata['source']} \n\n page_label: {doc.metadata['page_label']})"
        for doc in docs
    )


def format_response(obj: LLMResponse) -> str:
    """Format the LLMResponse object into a string representation."""
    sources_str = "\n\n".join([f"{s.source} (p. {s.page_label})" for s in obj.sources])
    return f"{obj.answer}\n\n\n{sources_str}"

In [18]:
from operator import itemgetter

# Create structured LLM
structured_llm = remote_llm.with_structured_output(LLMResponse)
# RAG Prompt
rag_template: str = """
    <instructions>
    You are a helpful AI assistant. Answer the following question based primarily on the provided context.

    <context>{context}</context>
    <question>{question}</question>

    If the context doesn't contain relevant information, you may use your general knowledge to provide a 
    helpful response.
    </instructions>

    <guidelines>
    - Prioritize information from the provided context when available
    - Keep responses concise (maximum 3 sentences unless the user requests more detail)
    - Be accurate and relevant to the question asked
    - If you cannot answer based on context or knowledge, respond with "I don't know"
    - Respond naturally to greetings and casual conversation
    - Use a friendly, professional tone
    </guidelines>
    """
rag_prompt = ChatPromptTemplate.from_template(rag_template)
final_rag_chain = (
    {
        "context": retrieval_chain | format_docs,
        "question": itemgetter("question"),
    }
    | rag_prompt
    | remote_llm
    | StrOutputParser()
)

final_rag_chain_with_structure = (
    {
        "context": retrieval_chain | format_docs,
        "question": itemgetter("question"),
    }
    | rag_prompt
    | structured_llm
    | format_response
)

In [19]:
response = final_rag_chain.invoke({"question": question})
print(f"Question: {question}\nResponse:")
display(Markdown(response))

Question: What players are likely to exit Chelsea?
Response:


The players mentioned as likely to leave Chelsea this summer are João Félix, Raheem Sterling, Ben Chilwell, Renato Veiga, Axel Disasi, Carney Chukwuemeka and Christopher Nkunku.

#### Structured Output

In [20]:
response = final_rag_chain_with_structure.invoke({"question": question})
print(f"Question: {question}\nResponse:")
display(Markdown(response))

Question: What players are likely to exit Chelsea?
Response:


The players mentioned as likely to leave Chelsea this summer are João Félix, Raheem Sterling, Ben Chilwell, Renato Veiga, Axel Disasi, Carney Chukwuemeka and Christopher Nkunku.


Chelsea transfer news, rumours and gossip – Sky Sports (pages 9‑11) (p. 9)

Chelsea rumours and gossip – Sky Sports (pages 9‑11) (p. 11)

### 2. RAG FUSION

- `Rag Fusion` generates multiple queries and retrieves documents for each.

- It then uses Reciprocal Rank Fusion (RRF) to re-rank the retrieved documents.

- This process scores documents based on their rank across all searches, promoting the most consistently relevant ones to the top of the final list.

<br>

[![image.png](https://i.postimg.cc/MH50McSm/image.png)](https://postimg.cc/8fFfxzs7)

In [21]:
# RAG-Fusion: Related
template = """
    <system>
        <role>
        You are a helpful query reconstructor that generates multiple search queries based on a 
        single input query to improve document retrieval.
        </role>

        <instructions>
        Generate exactly 3 distinct search queries that would help find relevant information for the given question.

        Original question: {question}

        Your search queries should:
        - Be specific and use actual entities/terms from the original question
        - NOT use placeholders. Use the actual entities mentioned
        - Cover different angles or phrasings of the same information need
        - Be suitable for semantic search in documents
        - The generated queries should be on a new line
        </instructions>

        <outputs>
        Output:
        </outputs>
    </system>
    """
prompt_rag_fusion = ChatPromptTemplate.from_template(template)
generate_queries = (
    prompt_rag_fusion
    | remote_llm
    | StrOutputParser()
    | (lambda x: x.strip().split("\n"))
)

In [22]:
def reciprocal_rank_fusion(
    results: list[list], k: int = 60, num_results: int = 4
) -> list[tuple[Any, Any]]:
    """
    Apply Reciprocal Rank Fusion (RRF) to combine multiple ranked document lists.

    RRF is a method for combining multiple ranked lists of documents into a single
    ranked list. It assigns scores to documents based on their ranks across all
    input lists using the formula: 1 / (rank + k), where k is a constant that
    controls the influence of lower-ranked documents.

    Parameters
    ----------
    results : list[list]
        A list of lists, where each inner list contains ranked documents
        from different retrieval methods or queries.
    k : int, default=60
        The RRF constant parameter. Higher values reduce the difference
        between ranks, making the fusion less sensitive to rank differences.
        Typical values range from 10 to 100.
    num_results : int, default=4
        The number of top results to return after fusion.

    Returns
    -------
    list[tuple[Any, Any]]
        A list of tuples where each tuple contains:
        - First element: The document object (deserialized from JSON)
        - Second element: The fused score (float)
        Documents are sorted by fused score in descending order.

    Notes
    -----
    - Documents are serialized to JSON strings for deduplication
    - The RRF formula ensures that documents appearing in multiple lists
      get higher combined scores
    - Documents with better ranks (lower rank numbers) receive higher scores

    Examples
    --------
    >>> doc_lists = [[doc1, doc2], [doc2, doc3], [doc1, doc3]]
    >>> fused = reciprocal_rank_fusion(doc_lists, k=60)
    >>> # Returns documents ranked by their combined RRF scores
    """
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    sorted_docs = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return sorted_docs[:num_results]

In [23]:
question: str = "Who is Xavi Simmons?"
response = generate_queries.invoke({"question": question})
response

['Xavi Simmons biography and background  ',
 'Xavi Simmons professional football career details  ',
 'Xavi Simmons personal life and achievements']

In [24]:
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
question: str = "Who is Xavi Simmons?"

docs = retrieval_chain_rag_fusion.invoke({"question": question})
print(len(docs))
print(f"Question: {question}\n\n")
console.print(docs)

4
Question: Who is Xavi Simmons?




[
    (
        Document(
            metadata={
                'producer': 'Skia/PDF m138',
                'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
                'creationdate': '2025-07-25T18:28:43+00:00',
                'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, 
loans and contracts | Football News | Sky Sports',
                'moddate': '2025-07-25T18:28:43+00:00',
                'source': '../../data/chelsea_transfer_news.pdf',
                'total_pages': 12,
                'page': 4,
                'page_label': '5',
                '_id': 'dd72468c-201a-49e7-98d0-3ffc51d54e8d',
                '_collection_name': 'cfc_transfer_news'
            },
            page_content='actually solid.\n"The club’s poor season casts a heavy shadow over all the players. After
a very\nstrong season the year before, Xavi missed the chance to take the next step\nin his 
development."\nAdvertisement\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: Live updates and latest 
on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 5/17'
        ),
        0.04972677595628415
    ),
    (
        Document(
            metadata={
                'producer': 'Skia/PDF m138',
                'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
                'creationdate': '2025-07-25T18:28:43+00:00',
                'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, 
loans and contracts | Football News | Sky Sports',
                'moddate': '2025-07-25T18:28:43+00:00',
                'source': '../../data/chelsea_transfer_news.pdf',
                'total_pages': 12,
                'page': 4,
                'page_label': '5',
                '_id': 'f9fa6af5-af60-4023-9dd4-6b384ccbd5b1',
                '_collection_name': 'cfc_transfer_news'
            },
            page_content='"Xavi chose the move to Leipzig wisely, together with his camp. The goal was\nto take on 
a significant role at an ambitious, up-and-coming club.\xa0\n"Especially in his first year, he performed 
brilliantly — in great form, he became\na standout both at Leipzig and in the Bundesliga.\xa0\n"The permanent 
transfer to Leipzig had been discussed well in advance; it was\nno surprise. His contract, running until 2027, 
clearly indicates: Leipzig is not\nmeant to be the final step or final club.\xa0\n"Xavi sees Leipzig and the 
Bundesliga as a major opportunity for\ndevelopment. Now, together with his camp, he wants to take the next step 
up\nthe ladder. The Bundesliga has been good for him."\n24 Jul\n16:06\nHow did Simons perform last season?\nSky 
Germany’s RB Leipzig reporter Philipp Hinze:\xa0\n"Xavi’s last season was definitely not as poor as it initially 
seemed. His stats\nwere totally fine; he contributed several goal involvements. Unfortunately, a\nserious injury 
held him back — he tore his syndesmosis ligament and was\nsidelined for several months.\xa0\n"Xavi sees himself as 
a leader at Leipzig. He often puts too much pressure on\nhimself. In the end, the entire club disappointed. It was 
the worst Bundesliga\nseason in the club’s history.\xa0\n"As a result, Xavi’s performance was also described as 
insufficient. The whole\nteam underperformed badly. But in truth, Xavi’s attacking numbers were\nactually 
solid.\n"The club’s poor season casts a heavy shadow over all the players. After a very\nstrong season the year 
before, Xavi missed the chance to take the next step\nin his development."\nAdvertisement\n25/07/2025, 19:28 
Chelsea transfer news, rumours and gossip: Live updates and latest on 

In [ ]:
# def extract_metadata(docs: list[Document]) -> list[dict[str, Any]]:
#     """Extract metadata from a list of documents."""
#     result = list({(doc.metadata["source"], doc.metadata["page_label"]) for doc, _ in docs})
#     return [{"source": row[0], "page_label": row[1]} for row in result]


In [25]:
# RAG Prompt
rag_template: str = """
    <instructions>
    You are a helpful AI assistant. Answer the following question based primarily on the provided context.

    <context>{context}</context>
    <question>{question}</question>

    If the context doesn't contain relevant information, you may use your general knowledge to provide a 
    helpful response.
    </instructions>

    <guidelines>
    - Prioritize information from the provided context when available
    - Keep responses concise (maximum 3 sentences unless the user requests more detail)
    - Be accurate and relevant to the question asked
    - If you cannot answer based on context or knowledge, respond with "I don't know"
    - Respond naturally to greetings and casual conversation
    - Use a friendly, professional tone
    </guidelines>
    """
rag_prompt = ChatPromptTemplate.from_template(rag_template)
final_rag_chain = (
    {
        "context": retrieval_chain_rag_fusion,
        "question": itemgetter("question"),
    }
    | rag_prompt
    | remote_llm
    | StrOutputParser()
)

In [26]:
question: str = "Which clubs are interested in Dewsbury Hall?"

response = final_rag_chain.invoke({"question": question})
print(f"Question: {question}\n\nResponse:")
display(Markdown(response))

Question: Which clubs are interested in Dewsbury Hall?

Response:


Fulham have shown interest in signing Chelsea midfielder Kiernan Dewsbury‑Hall.

#### Add Sources To Responses

- It is important to include sources in the responses to provide context and credibility to the information presented.

In [ ]:
from langchain_core.runnables import RunnableLambda


def extract_context_and_metadata(
    docs_with_scores: list[tuple[Any, float]],
) -> dict[str, Any]:
    """Extract both context and metadata from RAG fusion output."""
    docs = [doc for doc, _ in docs_with_scores]
    context = "\n\n".join(
        f"(content: {doc.page_content}\nsource: {doc.metadata['source']}\npage_label: {doc.metadata['page_label']})"
        for doc in docs
    )
    result = list(
        {(doc.metadata["source"], doc.metadata["page_label"]) for doc in docs}
    )
    metadata = [{"source": row[0], "page_label": row[1]} for row in result]
    return {"context": context, "metadata": metadata}


def combine_with_question(inputs: dict[str, Any]) -> dict[str, Any]:
    """Combine retrieval results with the original question."""
    question = inputs["question"]
    retrieval_results = inputs["retrieval_results"]
    return {
        "question": question,
        "context": retrieval_results["context"],
        "metadata": retrieval_results["metadata"],
    }


# Updated chain
final_rag_chain_with_sources = (
    {
        "question": itemgetter("question"),
        # Generate related queries
        "retrieval_results": retrieval_chain_rag_fusion
        # Extract context and metadata (sources)
        | RunnableLambda(extract_context_and_metadata),
    }
    | RunnableLambda(combine_with_question)
    | {
        "llm_input": {
            "context": itemgetter("context"),
            "question": itemgetter("question"),
        },
        "metadata": itemgetter("metadata"),
    }
    | {
        "response": itemgetter("llm_input")
        | rag_prompt
        | remote_llm
        | (lambda x: x.content),
        "metadata": itemgetter("metadata"),
    }
)

In [ ]:
question: str = "Which clubs are interested in Dewsbury Hall?"
result = final_rag_chain_with_sources.invoke({"question": question})

print(f"Question: {question}\n\nResponse:")
display(Markdown(result["response"]))
console.print(result["metadata"])

In [ ]:
question: str = "Who is Hato Jorrel?"
result = final_rag_chain_with_sources.invoke({"question": question})

print(f"Question: {question}\n\nResponse:")
display(Markdown(result["response"]))
console.print(result["metadata"])

In [ ]:
question: str = "Are Chelsea pursuing Isak?"
result = final_rag_chain_with_sources.invoke({"question": question})

print(f"Question: {question}\n\nResponse:")
display(Markdown(result["response"]))
console.print(result["metadata"])

In [ ]:
# uvr -m streamlit run frontend/main.py

import time
from operator import itemgetter
from typing import Any, Generator
from uuid import uuid4

import streamlit as st
from langchain.load import dumps, loads
from langchain.prompts import ChatPromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableSerializable
from langchain_core.vectorstores.base import VectorStoreRetriever
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

from model_config import LocalModel, RemoteModel
from settings import refresh_settings

settings = refresh_settings()
model_str_remote: str = RemoteModel.LLAMA_3_3_70B_INSTRUCT
model_str_local: str = LocalModel.MISTRAL_7B_INSTRUCT_V0_3_Q4_0


# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,  # type: ignore
    temperature=0.0,
    model=model_str_remote,  # type: ignore
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OLLAMA_URL,  # type: ignore
    temperature=0.0,
    model=model_str_local,  # type: ignore
)

llm = remote_llm


def get_document_splits(filepath: str) -> list[Any]:
    """Load a PDF file and split it into smaller chunks."""
    loader = PyPDFLoader(filepath)
    docs = loader.load()

    # Split the documents into smaller chunks
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=500, chunk_overlap=100)
    return text_splitter.split_documents(docs)


def get_vector_store(splits: list[Any], collection_name: str) -> VectorStoreRetriever:
    """Create a vector store retriever from document splits."""
    emb_model = OllamaEmbeddings(
        model=LocalModel.MXBAI_EMBED_LARGE,
    )
    emb = emb_model.embed_documents(["Hello world"])
    emb_size: int = len(emb[0])

    client = QdrantClient(url=settings.QDRANT_URL)
    collection_exists_flag: bool = client.collection_exists(collection_name=collection_name)

    if not collection_exists_flag:
        client = QdrantClient(url=settings.QDRANT_URL)
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
        )

        # Vector store
        vector_store: QdrantVectorStore = QdrantVectorStore.from_documents(
            documents=splits,
            embedding=emb_model,
            collection_name=collection_name,
            ids=[str(uuid4()) for _ in range(len(splits))],
        )
    else:
        vector_store = QdrantVectorStore.from_existing_collection(
            embedding=emb_model,
            collection_name=collection_name,
            url=settings.QDRANT_URL,
        )

    return vector_store.as_retriever(search_kwargs={"k": 3})


def get_rag_fusion_generator(llm: ChatOpenAI) -> RunnableSerializable[dict, Any]:
    """Generate RAG fusion queries."""
    template = """
    <system>
    <role>
    You are a helpful assistant that generates multiple search queries based on a single input query.
    </role>

    <instructions>
    Generate a max of 3 search queries related to the question
    <question>{question}</question>
    </instructions>

    <outputs>
    Output:
    </outputs>

    </system>
    """
    prompt_rag_fusion = ChatPromptTemplate.from_template(template)
    return prompt_rag_fusion | llm | StrOutputParser() | (lambda x: x.strip().split("\n"))


def reciprocal_rank_fusion(results: list[list], k: int = 60, num_results: int = 4) -> list[tuple[Any, Any]]:
    """
    Apply Reciprocal Rank Fusion (RRF) to combine multiple ranked document lists.
    """
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    sorted_docs = [(loads(doc), score) for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)]
    return sorted_docs[:num_results]


def extract_context_and_metadata(
    docs_with_scores: list[tuple[Any, float]],
) -> dict[str, Any]:
    """Extract both context and metadata from RAG fusion output."""
    docs = [doc for doc, _ in docs_with_scores]
    context = "\n\n".join(
        f"(content: {doc.page_content}\nsource: {doc.metadata['source']}\npage_label: {doc.metadata['page_label']})"
        for doc in docs
    )
    result = list({(doc.metadata["source"], doc.metadata["page_label"]) for doc in docs})
    metadata = [{"source": row[0], "page_label": row[1]} for row in result]
    return {"context": context, "metadata": metadata}


def combine_with_question(inputs: dict[str, Any]) -> dict[str, Any]:
    """Combine retrieval results with the original question."""
    question = inputs["question"]
    retrieval_results = inputs["retrieval_results"]
    return {
        "question": question,
        "context": retrieval_results["context"],
        "metadata": retrieval_results["metadata"],
    }


def get_final_rag_pipeline(retrieval_chain: Any, llm_model: Any) -> RunnableSerializable[dict, Any]:
    """Generate the final RAG pipeline."""
    # RAG Prompt
    rag_template: str = """
    <instructions>
    You are a helpful AI assistant. Answer the following question based primarily on the provided context.
    
    <context>{context}</context>
    <question>{question}</question>
    
    If the context doesn't contain relevant information, you may use your general knowledge to provide a 
    helpful response.
    </instructions>

    <guidelines>
    - Prioritize information from the provided context when available
    - Keep responses concise (maximum 3 sentences unless the user requests more detail)
    - Be accurate and relevant to the question asked
    - If you cannot answer based on context or knowledge, respond with "I don't know"
    - Respond naturally to greetings and casual conversation
    - Use a friendly, professional tone
    - Do NOT include sources in your response - they will be added separately
    </guidelines>
    """
    rag_prompt = ChatPromptTemplate.from_template(rag_template)
    return (
        {
            "question": itemgetter("question"),
            # Generate related queries
            "retrieval_results": retrieval_chain
            # Extract context and metadata (sources)
            | RunnableLambda(extract_context_and_metadata),
        }
        | RunnableLambda(combine_with_question)
        | {
            "llm_input": {
                "context": itemgetter("context"),
                "question": itemgetter("question"),
            },
            "metadata": itemgetter("metadata"),
        }
        | {
            "response": itemgetter("llm_input") | rag_prompt | llm_model | (lambda x: x.content),
            "metadata": itemgetter("metadata"),
        }
    )


# Streamlit UI
st.title("Chat With Your Documents")

# Initialize session state
if "rag_chain" not in st.session_state:
    st.session_state.rag_chain = None
if "messages" not in st.session_state:
    st.session_state.messages = []

with st.sidebar:
    uploaded_file = st.file_uploader("Upload a PDF file", type=["pdf"])
    if uploaded_file:
        # Save uploaded file temporarily
        with open(f"temp_{uploaded_file.name}", "wb") as f:
            f.write(uploaded_file.read())

        with st.form(key="rag_form", clear_on_submit=True):
            collection_name = st.text_input("Collection Name", value="demo")
            submit_button = st.form_submit_button("Create Collection")

            if submit_button:
                # Process the document
                with st.spinner("Processing document..."):
                    splits = get_document_splits(f"temp_{uploaded_file.name}")
                    generate_queries = get_rag_fusion_generator(llm=llm)
                    # Create or fetch collection
                    retriever = get_vector_store(splits, collection_name)
                    retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
                    st.session_state.rag_chain = get_final_rag_pipeline(
                        retrieval_chain_rag_fusion,
                        llm_model=llm,
                    )
                    st.success(
                        f"Collection '{collection_name}' created successfully and processed {len(splits)} document chunks!"
                    )

    else:
        st.warning("Please create a collection first!")

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Accept user input
if prompt := st.chat_input("What's on your mind?"):
    if st.session_state.rag_chain is None:
        st.warning("Please upload a PDF file first!")
    else:
        # Add user message to chat history
        st.session_state.messages.append({"role": "user", "content": prompt})
        # Display user message in chat message container
        with st.chat_message("user"):
            st.markdown(prompt)

        # Display assistant response in chat message container
        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                # model_response = st.session_state.rag_chain.stream({"question": prompt})
                def response_generator() -> Generator[Any | str, Any, None]:
                    """Generate response chunks from the RAG chain."""
                    for chunk in st.session_state.rag_chain.stream({"question": prompt}):
                        yield str(chunk)

                response = st.write_stream(response_generator())

        # Add assistant response to chat history
        st.session_state.messages.append({"role": "assistant", "content": response})


In [ ]:
# how many players are chelsea fc interested in signing?
# Any news about Sterling?
# Who is Xavi and what club does he play for?
# What are the names of the reporters that wrote the blog or article about chelsea transfers?